# Recommender System

This notebook demonstrates user-based and item-based collaborative filtering recommender system using cosine similarity.
Missing values are represented by `'?'`.


In [1]:
import numpy as np
import pandas as pd

# Editable ratings matrix with '?' for missing values

k =5 

ratings_input = {
    "Product A": [k, 4, '?', 1, 3, '?', 2, '?'],
    "Product B": [3, '?', 4, 1, '?', 2, '?', k],
    "Product C": ['?', 2, k, 4, '?', 3, 1, '?'],
    "Product D": [1, 2, 2, '?', 4, '?', k, '?'],
    "Product E": [2, '?', '?', 3, 1, 4, '?', k],
    "Product F": ['?', 3, 1, 2, '?', k, 4, '?'],
    "Product G": [4, '?', 3, '?', 2, 1, '?', k],
    "Product H": ['?', k, '?', 4, 3, '?', 2, 1]
}

    
users = [f"User{i+1}" for i in range(8)]
ratings_df = pd.DataFrame(ratings_input, index=users)
ratings = ratings_df.replace('?', np.nan).astype(float)
ratings_display = ratings_df.copy()
ratings_display


,Product A,Product B,Product C,Product D,Product E,Product F,Product G,Product H
User1,5,3,?,1,2,?,4,?
User2,4,?,2,2,?,3,?,5
User3,?,4,5,2,?,1,3,?
User4,1,1,4,?,3,2,?,4
User5,3,?,?,4,1,?,2,3
User6,?,2,3,?,4,5,1,?
User7,2,?,1,5,?,4,?,2
User8,?,5,?,?,5,?,5,1


## Cosine Similarity

Cosine similarity is computed using the formula:

$$
\cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \|\vec{B}\|}
$$

This formula directly computes the cosine of the angle between two vectors without measuring the angle itself.
It is efficient and works well with sparse data by ignoring missing values.

##  Computational Benefits of Cosine Similarity

Efficient: Uses dot products and norms.

Sparse-friendly: Ignores missing values.

Pattern-focused: Captures rating behavior.

Scalable: Works well with large datasets.

In [2]:
def cosine_similarity_nan(vec1, vec2):
    mask = ~np.isnan(vec1) & ~np.isnan(vec2)
    if np.sum(mask) == 0:
        return 0
    v1, v2 = vec1[mask], vec2[mask]
    dot = np.dot(v1, v2)
    norm1, norm2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return dot / (norm1 * norm2) if norm1 > 0 and norm2 > 0 else 0


## User Similarity Matrix
We compute pairwise cosine similarity between users based on their ratings.

In [3]:
user_sim = pd.DataFrame(np.ones((8, 8)), index=users, columns=users)
for u1 in users:
    for u2 in users:
        if u1 != u2:
            user_sim.loc[u1, u2] = cosine_similarity_nan(ratings.loc[u1].values, ratings.loc[u2].values)
user_sim


,User1,User2,User3,User4,User5,User6,User7,User8
User1,1.000000,0.964764,0.946864,0.684762,0.780654,0.729397,0.546268,0.964901
User2,0.964764,1.000000,0.752773,0.850131,0.894792,0.998868,0.779920,1.000000
User3,0.946864,0.752773,1.000000,0.875466,0.868243,0.695095,0.535264,0.989949
User4,0.684762,0.850131,0.875466,1.000000,0.809858,0.894427,0.723356,0.659082
User5,0.780654,0.894792,0.868243,0.809858,1.000000,0.650791,0.955330,0.673633
User6,0.729397,0.998868,0.695095,0.894427,0.650791,1.000000,0.956674,0.881917
User7,0.546268,0.779920,0.535264,0.723356,0.955330,0.956674,1.000000,1.000000
User8,0.964901,1.000000,0.989949,0.659082,0.673633,0.881917,1.000000,1.000000


## Item Similarity Matrix (Product Similarity Matrix)
We compute pairwise cosine similarity between items based on user ratings.

In [4]:
items = ratings.columns
item_sim = pd.DataFrame(np.ones((8, 8)), index=items, columns=items)
for i1 in items:
    for i2 in items:
        if i1 != i2:
            item_sim.loc[i1, i2] = cosine_similarity_nan(ratings[i1].values, ratings[i2].values)
item_sim

,Product A,Product B,Product C,Product D,Product E,Product F,Product G,Product H
Product A,1.000000,0.992278,0.666667,0.702251,0.722806,0.891485,0.997054,0.919272
Product B,0.992278,1.000000,0.925820,0.983870,0.915209,0.637455,0.971825,0.428086
Product C,0.666667,0.925820,1.000000,0.603860,0.960000,0.690909,0.976187,0.910840
Product D,0.702251,0.983870,0.603860,1.000000,0.650791,0.955904,0.729397,0.773841
Product E,0.722806,0.915209,0.960000,0.650791,1.000000,0.965616,0.847826,0.662994
Product F,0.891485,0.637455,0.690909,0.955904,0.965616,1.000000,0.496139,0.858137
Product G,0.997054,0.971825,0.976187,0.729397,0.847826,0.496139,1.000000,0.645942
Product H,0.919272,0.428086,0.910840,0.773841,0.662994,0.858137,0.645942,1.000000


## User-Based Predictions

Predicted rating of User $u$ for Product $i$ is calculated as follows:

$$
\hat{r}_{ui} = \frac{\sum\limits_{v\neq u} \text{sim}(u, v) \cdot r_{vi}}{\sum\limits_{v \neq u} |\text{sim}(u, v)|}
$$

## Why Is Modulus Used in the Computation?
 
The modulus (absolute value) is used in the denominator of the prediction formula to ensure that negative similarities (if any) still contribute to the total weight.
 
It prevents division by zero.

It maintains numerical stability and proper scaling of predictions.


## Example (Explicit calculation):

In the input ratings, there is no rating by User 1 for Product C. This rating is calculated as a weighted average of the ratings of other users who have rated Product C, that is, User 2, User 3, User 4, User 6 and User 7.

$$
\hat{r}_{1C} = \frac{\sum\limits_{v\neq 1} \text{sim}(1, v) \cdot r_{vC}}{\sum\limits_{v \neq 1} |\text{sim}(1, v)|}
$$

$$
\hat{r}_{1C} = \frac{(0.964764*2)+(0.946864*5)+(0.684762*4)+(0.729397*3)+(0.546268*1)}{0.964764+0.946864+0.684762+0.729397+0.546268}
$$

$$
\hat{r}_{1C} = 3.13460
$$
<br><br>

The predictions for the other missing ratings are calculated in a similar way.

In [5]:
user_recs = {}
for user in users:
    scores = {}
    for item in items:
        if np.isnan(ratings.loc[user, item]):
            total_sim, weighted_sum = 0, 0
            for other in users:
                if other != user and not np.isnan(ratings.loc[other, item]):
                    sim = user_sim.loc[user, other]
                    weighted_sum += sim * ratings.loc[other, item]
                    total_sim += abs(sim)
            if total_sim > 0:
                scores[item] = weighted_sum / total_sim
    if scores:
        print(f"{user} predictions: {scores}")
        best_item = max(scores, key=scores.get)
        user_recs[user] = (best_item, scores[best_item])
pd.DataFrame(user_recs, index=["Recommended Item", "Predicted Rating"]).T


User1 predictions: {'Product C': 3.134603282755995, 'Product F': 2.8519056936781717, 'Product H': 3.035069130297181}
User2 predictions: {'Product B': 3.0117468718007854, 'Product E': 3.0519308618421217, 'Product G': 3.01566514231657}
User3 predictions: {'Product A': 3.090560656571925, 'Product E': 2.9980902742709046, 'Product H': 2.9666427965485545}
User4 predictions: {'Product D': 2.7873623780510903, 'Product G': 2.8481529996744444}
User5 predictions: {'Product B': 2.9854623169705308, 'Product C': 2.937997717595245, 'Product F': 2.930741696396959}
User6 predictions: {'Product A': 2.931948984203489, 'Product D': 2.853970891891595, 'Product H': 3.0391667813447074}
User7 predictions: {'Product B': 3.035059745874799, 'Product E': 3.1195098107254657, 'Product G': 2.9192669731332432}
User8 predictions: {'Product A': 3.142320397917523, 'Product C': 2.920321549056414, 'Product D': 2.7307716676852842, 'Product F': 3.0275556370837644}


,Recommended Item,Predicted Rating
User1,Product C,3.134603
User2,Product E,3.051931
User3,Product A,3.090561
User4,Product G,2.848153
User5,Product B,2.985462
User6,Product H,3.039167
User7,Product E,3.11951
User8,Product A,3.14232


## Item-Based Predictions (Product-Based Predictions)

Predicted rating of User $u$ for Product $i$ is calculated as follows:

$$
\hat{r}_{ui} = \frac{\sum\limits_{j\neq i} \text{sim}(i, j) \cdot r_{uj}}{\sum\limits_{j\neq i} |\text{sim}(i, j)|}
$$




## Example (Explicit calculation):

In the input ratings, there is no rating by User 1 for Product C. This rating is calculated as a weighted average of the ratings of User 1 for other products he/she has rated, that is, Product A, Product B, Product D, Product E and Product G.

$$
\hat{r}_{1C} = \frac{\sum\limits_{j\neq C} \text{sim}(C, j) \cdot r_{1j}}{\sum\limits_{j \neq C} |\text{sim}(C, j)|}
$$

$$
\hat{r}_{1C} = \frac{(0.666667*5)+(0.925820*3)+(0.603860*1)+(0.960000*2)+(0.976187*4)}{0.666667+0.925820+0.603860+0.960000+0.976187}
$$

$$
\hat{r}_{1C} = 3.0343133
$$
<br><br>

The predictions for the other missing ratings are calculated in a similar way.

In [6]:
item_recs = {}
for user in users:
    scores = {}
    for item in items:
        if np.isnan(ratings.loc[user, item]):
            total_sim, weighted_sum = 0, 0
            for rated_item in items:
                if not np.isnan(ratings.loc[user, rated_item]) and item != rated_item:
                    sim = item_sim.loc[item, rated_item]
                    weighted_sum += sim * ratings.loc[user, rated_item]
                    total_sim += abs(sim)
            if total_sim > 0:
                scores[item] = weighted_sum / total_sim
    if scores:
        print(f"{user} predictions: {scores}")
        best_item = max(scores, key=scores.get)
        item_recs[user] = (best_item, scores[best_item])
pd.DataFrame(item_recs, index=["Recommended Item", "Predicted Rating"]).T


User1 predictions: {'Product C': 3.0343133612725537, 'Product F': 2.8483972419083963, 'Product H': 3.0798250633631996}
User2 predictions: {'Product B': 2.984564764409346, 'Product E': 3.1105449657993858, 'Product G': 3.151728908817669}
User3 predictions: {'Product A': 2.962442462232778, 'Product E': 3.0583452316966, 'Product H': 2.9335476096789224}
User4 predictions: {'Product D': 2.368283101818178, 'Product G': 2.4302362755629745}
User5 predictions: {'Product B': 2.5762621598223, 'Product C': 2.4432792538988606, 'Product F': 2.6469001308541102}
User6 predictions: {'Product A': 2.887452489049005, 'Product D': 3.030566199127504, 'Product H': 3.1880479714760037}
User7 predictions: {'Product B': 2.8319325147859047, 'Product E': 2.7378730183874187, 'Product G': 2.5733269287491507}
User8 predictions: {'Product A': 3.987421284243715, 'Product C': 4.034320752438157, 'Product D': 4.013554957014297, 'Product F': 3.839315444650587}


,Recommended Item,Predicted Rating
User1,Product H,3.079825
User2,Product G,3.151729
User3,Product E,3.058345
User4,Product G,2.430236
User5,Product F,2.6469
User6,Product H,3.188048
User7,Product B,2.831933
User8,Product C,4.034321


## EXERCISES

The digit root of a number is a single-digit number obtained by repeatedly summing the digits of a number until a single-digit result is achieved. 

Take $R = \text{Digit root of your Roll Number}$ and $k = (R \mod 4) + 1$.

For this value of $k$, implement the Recommender System which includes 

1. one explicit calculation of cosine similarity between two users. Verify your answer.

2. one explicit calculation of cosine similarity between two products. Verify your answer.

3. one explicit calculation of user-based prediction. Verify your answer.

4. one explicit calculation of product-based prediction. Verify your answer.

## NOTE:

Cosine similarity is also widely used in Natural Language Processing (NLP) to measure how similar two text (word) vectors are, independent of their magnitude. It helps capture how close two vectors are in high-dimensional space, reflecting the similarity of meanings between words, sentences, or documents.

In [10]:
k=2
ua, ub = "User1", "User2"
a, b = ratings.loc[ua].values, ratings.loc[ub].values
m = ~np.isnan(a) & ~np.isnan(b)
manual = np.sum(a[m] * b[m]) / (np.sqrt(np.sum(a[m]**2)) * np.sqrt(np.sum(b[m]**2)))
print("1) Manual:", round(manual, 6), "| Code:", round(user_sim.loc[ua, ub], 6), "|", np.isclose(manual, user_sim.loc[ua, ub]))

ia, ib = "Product A", "Product B"
a, b = ratings[ia].values, ratings[ib].values
m = ~np.isnan(a) & ~np.isnan(b)
manual = np.sum(a[m] * b[m]) / (np.sqrt(np.sum(a[m]**2)) * np.sqrt(np.sum(b[m]**2)))
print("2) Manual:", round(manual, 6), "| Code:", round(item_sim.loc[ia, ib], 6), "|", np.isclose(manual, item_sim.loc[ia, ib]))

u, i = "User1", "Product C"
others = [v for v in users if v != u and not np.isnan(ratings.loc[v, i])]
manual = sum(user_sim.loc[u, v] * ratings.loc[v, i] for v in others) / sum(abs(user_sim.loc[u, v]) for v in others)
code = dict(user_recs_all := {}) or None
print("3) Manual:", round(manual, 6))

js = [j for j in items if j != i and not np.isnan(ratings.loc[u, j])]
manual_ib = sum(item_sim.loc[i, j] * ratings.loc[u, j] for j in js) / sum(abs(item_sim.loc[i, j]) for j in js)
print("4) Manual:", round(manual_ib, 6))

1) Manual: 1.0 | Code: 1.0 | True
2) Manual: 0.989949 | Code: 0.989949 | True
3) Manual: 2.313706
4) Manual: 2.327644


In [11]:
ua, ub = "User1", "User2"
a, b = ratings.loc[ua].values, ratings.loc[ub].values
m = ~np.isnan(a) & ~np.isnan(b)
manual = np.sum(a[m] * b[m]) / (np.sqrt(np.sum(a[m]**2)) * np.sqrt(np.sum(b[m]**2)))
print("1) Manual:", round(manual, 6), "| Code:", round(user_sim.loc[ua, ub], 6), "|", np.isclose(manual, user_sim.loc[ua, ub]))

ia, ib = "Product A", "Product B"
a, b = ratings[ia].values, ratings[ib].values
m = ~np.isnan(a) & ~np.isnan(b)
manual = np.sum(a[m] * b[m]) / (np.sqrt(np.sum(a[m]**2)) * np.sqrt(np.sum(b[m]**2)))
print("2) Manual:", round(manual, 6), "| Code:", round(item_sim.loc[ia, ib], 6), "|", np.isclose(manual, item_sim.loc[ia, ib]))

u, i = "User1", "Product C"

vs = [v for v in users if v != u and not np.isnan(ratings.loc[v, i])]
manual = sum(user_sim.loc[u, v] * ratings.loc[v, i] for v in vs) / sum(abs(user_sim.loc[u, v]) for v in vs)
ws, tot = 0, 0
for v in users:
    if v != u and not np.isnan(ratings.loc[v, i]):
        ws += user_sim.loc[u, v] * ratings.loc[v, i]
        tot += abs(user_sim.loc[u, v])
print("3) Manual:", round(manual, 6), "| Notebook formula:", round(ws / tot, 6), "|", np.isclose(manual, ws / tot))

js = [j for j in items if j != i and not np.isnan(ratings.loc[u, j])]
manual = sum(item_sim.loc[i, j] * ratings.loc[u, j] for j in js) / sum(abs(item_sim.loc[i, j]) for j in js)
ws, tot = 0, 0
for j in items:
    if j != i and not np.isnan(ratings.loc[u, j]):
        ws += item_sim.loc[i, j] * ratings.loc[u, j]
        tot += abs(item_sim.loc[i, j])
print("4) Manual:", round(manual, 6), "| Notebook formula:", round(ws / tot, 6), "|", np.isclose(manual, ws / tot))

1) Manual: 1.0 | Code: 1.0 | True
2) Manual: 0.989949 | Code: 0.989949 | True
3) Manual: 2.313706 | Notebook formula: 2.313706 | True
4) Manual: 2.327644 | Notebook formula: 2.327644 | True
